# C4 · Ajuste de PSF (psffit)

**Spec:** [`docs/spec_C4_codex_psf_fitting.md`](../docs/spec_C4_codex_psf_fitting.md)  |  **Bloque:** C · Extracción  |  **Run por defecto:** `ROXs12b_realigned`

Ajusta simultáneamente estrella + compañero sobre el modelo de PSF (método canónico).

| | |
|---|---|
| **Entrada** | Cubo + PSF (C1) |
| **Salida (QC/productos)** | `stages/spec_psffit_qc.json` |
| **Consume aguas abajo** | D1, D2, E1, E3 (método canónico) |


## Qué hace C4 y por qué es el canónico

C4 (`psffit`) es el **método primario** recomendado por la literatura para un compañero a ~1.75″. Por canal ajusta un modelo **lineal** por mínimos cuadrados:

```
D(y,x) = a·P_estrella + b·P_compañero + (c₀ + c₁·y + c₂·x)
```

Las incógnitas por canal son solo `(a, b, c₀, c₁, c₂)`: `a` amplitud de la estrella, `b` la del compañero, y `(c₀,c₁,c₂)` un **plano de fondo local**. Toda la no-linealidad (forma de PSF, posiciones) quedó resuelta aguas arriba (C1/B3) → el problema es lineal, exacto y rápido. **Esa es la decisión de diseño central.**

**Por qué es el canónico:** ajusta estrella + compañero + fondo **simultáneamente**, atacando la decontaminación del halo de frente, **sin sustracción agresiva**. El plano `(c₀,c₁,c₂)` absorbe el gradiente del halo → es el método **menos sesgado** (por eso su media de controles es la más pequeña; ver el trabajo de referenciación en C3/D2).

**Calidad del ajuste:** χ²ᵣ ≈ 1.03 (excelente), número de condición 10.7 (bien condicionado), `rho_ab` 0.17 (estrella y compañero **separables**), crosstalk 0.03 (las líneas de la estrella no contaminan al compañero), y la estrella se recupera (flujo psffit / apertura grande = 1.014).

**Salvedad:** `rho_bc` mediana 0.45 (1303 canales >0.5) — el compañero es débil y queda parcialmente degenerado con el plano de fondo. Errores empíricos (M5 rojo). El producto final (`spec_final_object`) lleva además `cont_runmed_biasref` (referenciado a controles, del trabajo de D2) para G3.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x03_psffit.sh --run-id $RUN
```

Moderado (~3 min con Psfao lru_cache; sin cache era 2h+).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_psffit_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x03_psffit.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_psffit_qc.json', RUN_ID)
nb.show(qc, keys=['chi2r.median', 'condition_number_median', 'rho_ab_median', 'vs_large_aperture_median_ratio', 'v3_star_scale_ok'], title='C4')


## Resultados que llevaron a la conclusión

Región de ajuste, condicionamiento, χ²ᵣ, crosstalk y validación de la estrella del `spec_psffit_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C4', 'stages/spec_psffit_qc.json'):
        q = nb.load_qc('stages/spec_psffit_qc.json', RUN_ID)
        fr = q['fit_region']; cond = q['conditioning']; ck = q['checks']
        print(f"región: estrella r={fr['star_radius_px']:.0f}px, compañero r={fr['comp_radius_px']:.0f}px, {fr['n_pixels_median']:.0f} px/ajuste")
        print(f"χ²ᵣ: mediana {q['chi2r']['median']:.3f} (p90 {q['chi2r']['p90']:.2f})")
        print(f"condicionamiento: cond={cond['condition_number_median']:.1f}, "
              f"rho_ab(estrella-compañero)={cond['rho_ab_median']:.2f}, "
              f"rho_bc(compañero-fondo)={cond['rho_bc_median']:.2f} ({cond['channels_rho_bc_gt_0p5']} canales >0.5)")
        print(f"crosstalk (líneas estrella->compañero) = {q['crosstalk']['metric_corr_b_vs_a_lines']:.3f}")
        print(f"estrella recuperada (psffit/apertura grande) = {q['star_product_check']['vs_large_aperture_median_ratio']:.3f}")
        print(f"checks: v3_star_scale={ck['v3_star_scale_ok']} v4_rho_ab={ck['v4_rho_ab_ok']} rho_bc_warning={ck['rho_bc_warning']}")
        print(f"errores: {q['errors']['mode']} ({q['errors']['n_controls']} controles); open_issue: {q['open_issues'][0]}")


## Plot 1 — el cubo residual: estrella + compañero removidos

Entrada (`stage02`) vs residual del ajuste (`cube_psffit_residual.fits`), colapsados en 7000–8500 Å. El halo estelar casi desaparece (queda solo el residuo del anillo ~4–5% de C1 en el núcleo) y el compañero también → demuestra la decontaminación **simultánea** y el χ²ᵣ≈1. `+` estrella, `○` compañero.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    from musepipe.io import read_wavelength_axis
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/spec_psffit_qc.json', RUN_ID)
    loc = nb.load_qc('stages/stage01c_qc.json', RUN_ID)   # posiciones oficiales (B3)
    (py, px), (cy, cx) = loc['primary']['pos_yx'], loc['companion']['pos_yx']
    def cube(path):
        h = fits.open(path); hd = next(x for x in h if x.data is not None)
        d = np.asarray(hd.data, float); h.close(); return d[0] if d.ndim == 4 else d
    res = cube(q['products']['residual_cube'])
    inp = cube(rd / 'stages' / 'stage02_xcorr_cube_stack.fits')
    with fits.open(rd / 'stages' / 'stage02_xcorr_cube_stack.fits') as _h:
        wave = read_wavelength_axis(_h)   # ext WAVELENGTH del stack
    sel = (wave >= 7000) & (wave <= 8500)
    imgi = np.nanmedian(inp[sel], axis=0); imgr = np.nanmedian(res[sel], axis=0)
    v = np.nanpercentile(imgi, [30, 99.5])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
    for ax, im, t in [(axes[0], imgi, 'entrada: estrella + compañero'),
                      (axes[1], imgr, 'residual psffit: ambos removidos (χ²ᵣ≈1)')]:
        ax.imshow(im, origin='lower', cmap='magma', vmin=v[0], vmax=v[1])
        ax.plot(px, py, '+', color='cyan', ms=10); ax.plot(cx, cy, 'o', mfc='none', mec='lime', ms=12)
        ax.set_title(t); ax.axis('off')
    fig.tight_layout()
    outdir = rd / 'plots' / 'c4_psffit'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'residual.png', dpi=110); print('figura ->', outdir / 'residual.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — el espectro canónico del compañero (no-detección)

`spec_psffit_object.fits` con la banda ±1σ empírica de los controles y Hα. El continuo sube al rojo (SED real de enana fría) y **no hay nada en Hα** — la no-detección con el método canónico. El azul (λ<7000) está dominado por ruido (SNR<1).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_psffit_object.fits')
    wave = np.asarray(h[1].data['wave_A'], float)   # eje λ del propio producto
    flux = np.asarray(h[1].data['flux'], float); h.close()
    C = np.load(rd / 'stages' / 'spec_psffit_controls.npz')['control_spectra']
    sig = np.nanstd(C, axis=0)
    sm = np.convolve(np.nan_to_num(flux), np.ones(41) / 41, mode='same')
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(wave, -sig, sig, color='0.85', label=f'±1σ empírico ({C.shape[0]} controles)')
    ax.plot(wave, flux, lw=0.3, color='0.55', alpha=0.6)
    ax.plot(wave, sm, lw=1.3, color='tab:blue', label='flujo compañero psffit (suavizado)')
    ax.axvline(6563, color='tab:red', ls=':', label='Hα'); ax.axhline(0, color='0.6', lw=0.6)
    ax.set_ylim(np.nanpercentile(flux, 2), np.nanpercentile(flux, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (canónico)')
    ax.set_title('C4 · espectro canónico psffit del compañero — nada en Hα (no-detección)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c4_psffit'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'spectrum.png', dpi=110); print('figura ->', outdir / 'spectrum.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Ajuste lineal por canal** `a·P_estrella + b·P_compañero + plano`: toda la no-linealidad se resuelve aguas arriba (C1/B3). Es el método primario de la literatura.
- **Simultáneo estrella+compañero+plano**: maneja el gradiente del halo de frente, sin sobre-sustracción; positivo y físico en el borde → **método canónico**. · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)
- Ajuste sano: χ²ᵣ≈1.03, cond 10.7, rho_ab 0.17 (separables), estrella recuperada 1.014, crosstalk 0.03.
- Salvedad: `rho_bc`≈0.45 (1303 canales >0.5), compañero débil parcialmente degenerado con el plano; errores empíricos (M5 rojo).


## Conclusión (registrada)

**C4: método canónico `psffit`; ajuste lineal simultáneo estrella+compañero+plano; χ²ᵣ≈1.03; residual limpio; no-detección.**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Ajuste:** por canal `a·P★ + b·P_c + (c₀+c₁y+c₂x)`; región estrella 20px / compañero 12px; 1705 px/ajuste.
- **Validación:** estrella recuperada (1.014 vs apertura grande), crosstalk 0.03, rho_ab 0.17 (separables).
- **Residual:** halo + compañero removidos (queda el anillo ~4–5% de C1).
- **Salvedad:** rho_bc≈0.45 (degeneración compañero-fondo en canales débiles); errores empíricos.
- **Downstream:** es el canónico de D1/D2/E1/E3; el par primario de D1 es psffit vs optimal_psfsub. El producto final lleva `cont_runmed_biasref` para G3.
